# AdaBoost Regressor — Food Delivery Time Prediction

This notebook adds an AdaBoost regression model to the food delivery time project. It uses the same 80/20 train/test split and reports MAE, MSE, RMSE, and R². Hyperparameter selection uses cross-validation on the training split; the held-out test split is used only for final evaluation.

The target is `Time_taken_min`. `Average_Speed_kmph` is excluded because it can reveal information from a completed delivery. Order identifiers and the raw date are excluded too.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
DATA_PATH = Path(r'D:\innomatics\Machine Learning\food delivary time prediction\Food_Delivery_Time_Prediction.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('Food_Delivery_Time_Prediction.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Place Food_Delivery_Time_Prediction.csv beside this notebook or update DATA_PATH.')

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
display(df.head())

In [ ]:
target = 'Time_taken_min'
excluded = [col for col in ['Average_Speed_kmph', 'Order_ID', 'Order_Date'] if col in df.columns]
if target not in df.columns:
    raise KeyError(f'Required target column {target!r} is missing.')

X = df.drop(columns=[target, *excluded]).copy()
y = pd.to_numeric(df[target], errors='coerce')
valid_rows = y.notna() & np.isfinite(y)
X, y = X.loc[valid_rows].copy(), y.loc[valid_rows].astype(float)
print('Excluded columns:', excluded)
print('Feature columns:', X.columns.tolist())
print('Rows available:', len(X))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

ordinal_categories = {
    'Restaurant_Load': ['Low', 'Medium', 'High'],
    'Delivery_Distance_Category': ['Short', 'Medium', 'Long'],
    'Traffic_Level': ['Low', 'Moderate', 'High', 'Severe'],
    'Delivery_Priority': ['Normal', 'Priority', 'VIP'],
}
ordinal_cols = [col for col in ordinal_categories if col in X_train.columns]
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
nominal_cols = [col for col in X_train.columns if col not in numeric_cols + ordinal_cols]
print('Numerical:', numeric_cols)
print('Ordinal:', ordinal_cols)
print('Nominal:', nominal_cols)

In [ ]:
transformers = []
if numeric_cols:
    transformers.append(('numeric', SimpleImputer(strategy='median'), numeric_cols))
if nominal_cols:
    nominal_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    transformers.append(('nominal', nominal_pipe, nominal_cols))
if ordinal_cols:
    ordinal_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(
            categories=[ordinal_categories[col] for col in ordinal_cols],
            handle_unknown='use_encoded_value', unknown_value=-1
        )),
    ])
    transformers.append(('ordinal', ordinal_pipe, ordinal_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
base_tree = DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE)
ada_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', AdaBoostRegressor(estimator=base_tree, random_state=RANDOM_STATE))
])
ada_pipeline

## Baseline AdaBoost

AdaBoost trains a sequence of weak regressors, giving more weight to observations that earlier trees predicted poorly. The default base estimator here is a small decision tree.

In [ ]:
baseline_model = ada_pipeline.set_params(
    model__n_estimators=100, model__learning_rate=0.05,
    model__estimator__max_depth=3, model__loss='linear'
)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mean_squared_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R²': r2_score(y_true, y_pred),
    }

baseline_metrics = regression_metrics(y_test, y_pred_baseline)
print('Baseline AdaBoost results')
pd.Series(baseline_metrics).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred_baseline, alpha=0.35)
limits = [min(y_test.min(), y_pred_baseline.min()), max(y_test.max(), y_pred_baseline.max())]
axes[0].plot(limits, limits, '--', color='red')
axes[0].set(xlabel='Actual delivery time (min)', ylabel='Predicted delivery time (min)', title='Baseline: actual vs predicted')
axes[1].scatter(y_pred_baseline, y_test - y_pred_baseline, alpha=0.35)
axes[1].axhline(0, linestyle='--', color='red')
axes[1].set(xlabel='Predicted delivery time (min)', ylabel='Residual (actual - predicted)', title='Baseline residuals')
plt.tight_layout()
plt.show()

## Tune the model

Randomized search chooses hyperparameters by 3-fold cross-validated MAE on the training split. The test data remains held out until the final comparison.

In [ ]:
param_distributions = {
    'model__n_estimators': [50, 100, 150, 250],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.5],
    'model__estimator__max_depth': [1, 2, 3, 4],
    'model__estimator__min_samples_leaf': [1, 2, 5],
    'model__loss': ['linear', 'square', 'exponential'],
}

random_search = RandomizedSearchCV(
    estimator=ada_pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)
random_search.fit(X_train, y_train)
print('Best parameters:', random_search.best_params_)
print(f'Best 3-fold CV MAE: {-random_search.best_score_:.3f} min')

In [ ]:
best_ada = random_search.best_estimator_
y_pred_tuned = best_ada.predict(X_test)
tuned_metrics = regression_metrics(y_test, y_pred_tuned)
comparison = pd.DataFrame(
    [baseline_metrics, tuned_metrics],
    index=['Baseline AdaBoost', 'Tuned AdaBoost']
)
display(comparison.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred_tuned, alpha=0.35)
limits = [min(y_test.min(), y_pred_tuned.min()), max(y_test.max(), y_pred_tuned.max())]
axes[0].plot(limits, limits, '--', color='red')
axes[0].set(xlabel='Actual delivery time (min)', ylabel='Predicted delivery time (min)', title='Tuned: actual vs predicted')
axes[1].scatter(y_pred_tuned, y_test - y_pred_tuned, alpha=0.35)
axes[1].axhline(0, linestyle='--', color='red')
axes[1].set(xlabel='Predicted delivery time (min)', ylabel='Residual (actual - predicted)', title='Tuned residuals')
plt.tight_layout()
plt.show()

## Feature importance and comparison

The model's feature importances are aggregated across its fitted decision trees. Treat them as a summary, since impurity-based importance can favor features with many possible values. Compare test MAE and RMSE with your other model notebooks using the same feature set and split. Lower MAE/RMSE and higher R² indicate better held-out performance.

In [ ]:
feature_names = best_ada.named_steps['preprocessor'].get_feature_names_out()
importances = best_ada.named_steps['model'].feature_importances_
feature_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
display(feature_importance.head(20).to_frame('Importance'))
feature_importance.head(20).sort_values().plot(kind='barh', figsize=(9, 7), title='Top AdaBoost feature importances')
plt.xlabel('Impurity-based importance')
plt.tight_layout()
plt.show()